# load_openalex_author_institution_year

Prototipo del nodo `load_openalex_author_institution_year` del pipeline `load_openalex`. No guarda datasets.


In [ ]:
import pandas as pd
from pandas import json_normalize

%load_ext kedro.ipython


In [ ]:
df_author_raw = catalog.load('raw/openalex/author/parquet/author_dev')
df_author_raw.head(2)


In [ ]:
def _select_with_metadata(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = _add_openalex_extracted_metadata(df)
    return df.loc[:, [*columns, *_EXTRACTED_META_COLS]].copy()


In [ ]:
def _stringify_object_columns(
    df: pd.DataFrame,
    exclude_columns: list[str] | None = None,
) -> pd.DataFrame:
    exclude_columns = set(exclude_columns or [])
    for column in df.columns:
        if column in exclude_columns:
            continue
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].where(df[column].notna(), pd.NA).astype("string")
    return df


In [ ]:
def _serialize_nested_value(value):
    if value is None or value is pd.NA:
        return value
    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


In [ ]:
def _serialize_nested_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(_serialize_nested_value)
    return df


In [ ]:
def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df["extract_datetime"] = pd.to_datetime(df["extract_datetime"], errors="coerce")
    df["_extract_datetime"] = pd.to_datetime(df["_extract_datetime"], errors="coerce")
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date
    return df


In [ ]:
def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = _serialize_nested_columns(df)
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    load_datetime = pd.to_datetime(load_datetime)
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openalex_author_institution_year(df: pd.DataFrame)-> pd.DataFrame:

    # Selecciono columna con id de author y afiliación
    df_author = _select_with_metadata(df, ['id', 'affiliations'])
    df_author = df_author.convert_dtypes()

    # Proceso columna 'affiliations'
    df_author = df_author.explode('affiliations').reset_index(drop=True)
    affiliation_expanded = pd.json_normalize(df_author["affiliations"])
    affiliation_expanded = affiliation_expanded.loc[:,['institution.id','years']]

    df_author2affiliation = pd.concat([df_author, affiliation_expanded], axis=1)
    df_author2affiliation.drop(columns=["affiliations"], inplace=True)
    df_author2affiliation = df_author2affiliation.explode('years')

    df_author2affiliation.rename(columns={'institution.id':'institution_id'}, inplace=True)
    df_author2affiliation = df_author2affiliation.convert_dtypes()

    df_author2affiliation = _add_openalex_loaded_metadata(df_author2affiliation)
    
    return df_author2affiliation


In [ ]:
df_author2affiliation = load_openalex_author_institution_year(df_author_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_author2affiliation', 'rows': len(df_author2affiliation), 'columns': len(df_author2affiliation.columns)}])


In [ ]:
df_author2affiliation.head(2)
